# 附录：用最少代码训练一个“问候回复模型”

这个小实验的目标很简单：

```text
输入：你好
输出：你好呀，很高兴见到你！
```

它不是 GPT-2 本体，但思想和 GPT-2 很像：**根据前面的文字，预测下一个字，不断预测下去，就生成了一句话。**

我们故意把模型做得很小，只训练它学会几种问候回复，方便课堂演示。

## 1. 准备一小批训练数据

格式统一成：

```text
用户：问题
助手：回答<结束>
```

模型会学习这种对话文本的规律。

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

pairs = [
    ('你好', '你好呀，很高兴见到你！'),
    ('hello', 'Hello，很高兴见到你！'),
    ('嗨', '嗨！今天也要好好学习 AI。'),
    ('早上好', '早上好，祝你今天学习顺利！'),
    ('晚上好', '晚上好，今天辛苦了。'),
    ('你是谁', '我是一个很小的问候模型。'),
    ('你好啊', '你好啊，我在这里。'),
    ('hi', 'Hi，很高兴和你聊天。'),
    ('下午好', '下午好，我们继续学习。'),
    ('在吗', '在的，有什么想问我？'),
    ('哈喽', '哈喽，很高兴见到你。'),
    ('早', '早呀，准备开始学习吧。'),
    ('晚安', '晚安，祝你休息好。'),
    ('谢谢', '不客气，很高兴帮到你。'),
    ('再见', '再见，下次继续学习。'),
]

text = '\n'.join([f'用户：{q}\n助手：{a}<结束>' for q, a in pairs])
print(text[:120])

用户：你好
助手：你好呀，很高兴见到你！<结束>
用户：hello
助手：Hello，很高兴见到你！<结束>
用户：嗨
助手：嗨！今天也要好好学习 AI。<结束>
用户：早上好
助手：早上好，祝你今天学习顺利！<结束>
用户：晚上好
助手：


## 2. 把文字变成数字

神经网络不能直接吃文字，所以先把每个字符变成编号。

In [2]:
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}

data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)
vocab_size = len(chars)
block_size = 64

print('字符表大小:', vocab_size)
print('训练文本长度:', len(data))

字符表大小: 91
训练文本长度: 380


## 3. 构造训练批次

训练任务是：看到前面一串字符，预测下一个字符。

In [3]:
def get_batch(batch_size=32):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in ix])
    y = torch.stack([data[i+1:i+block_size+1] for i in ix])
    return x, y

x, y = get_batch(2)
print('输入形状:', x.shape)
print('目标形状:', y.shape)

输入形状: torch.Size([2, 64])
目标形状: torch.Size([2, 64])


## 4. 定义一个很小的 GPT-like 模型

GPT-2 的核心是 Transformer。这里我们只用两层很小的 Transformer，让它学会问候回复。

In [4]:
class TinyGPT(nn.Module):
    def __init__(self, vocab_size, n_emb=64, n_head=4, n_layer=2):
        super().__init__()
        self.token = nn.Embedding(vocab_size, n_emb)
        self.position = nn.Embedding(block_size, n_emb)
        layer = nn.TransformerEncoderLayer(n_emb, n_head, 4*n_emb, batch_first=True)
        self.transformer = nn.TransformerEncoder(layer, n_layer)
        self.head = nn.Linear(n_emb, vocab_size)

    def forward(self, x):
        B, T = x.shape
        h = self.token(x) + self.position(torch.arange(T))
        mask = torch.triu(torch.ones(T, T) * float('-inf'), diagonal=1)
        h = self.transformer(h, mask)
        return self.head(h)

model = TinyGPT(vocab_size)
print('参数量:', sum(p.numel() for p in model.parameters()))

参数量: 115803


## 5. 开始训练

损失越低，说明模型越会预测下一个字。

In [5]:
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-3)

for step in range(801):
    xb, yb = get_batch()
    logits = model(xb)
    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1))

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    if step % 200 == 0:
        print(step, round(loss.item(), 4))

0 4.5876


200 0.0647


400 0.047


600 0.0408


800 0.0425


## 6. 让模型回复一句话

生成时，只给它：

```text
用户：你好
助手：
```

然后让模型一个字一个字往下写，直到写到 `<结束>`。

In [6]:
@torch.no_grad()
def chat(message, max_new_tokens=60):
    prompt = f'用户：{message}\n助手：'
    idx = torch.tensor([[stoi[ch] for ch in prompt]], dtype=torch.long)

    for _ in range(max_new_tokens):
        logits = model(idx[:, -block_size:])
        probs = F.softmax(logits[:, -1, :], dim=-1)
        next_id = torch.multinomial(probs, num_samples=1)
        idx = torch.cat([idx, next_id], dim=1)

        out = ''.join(itos[i] for i in idx[0].tolist())
        if '<结束>' in out:
            break

    return out.split('助手：')[-1].split('<结束>')[0]

for q in ['你好', 'hello', '早上好', '在吗', '晚安']:
    print(q, '=>', chat(q))

你好 => 你好呀，很高兴见到你。
hello => Hello，很高兴见到你！
早上好 => 早上好，祝你今天学习顺利！
在吗 => 在的，有什么想问我？
晚安 => 晚安，祝你休息好。


## 7. 这个小模型说明了什么

它很小，只会回答训练数据附近的问题，但它已经具备大模型最核心的生成逻辑：

```text
上下文 → 预测下一个字 → 接着预测 → 形成回复
```

真正的 GPT-2 / GPT-4 区别在于：

- 数据大得多。
- 模型大得多。
- 训练时间长得多。
- 上下文更长。
- 学到的语言和知识更丰富。

但第一性原理是一样的：**模型不是提前写死回答，而是根据上下文一步步生成。**